## 1. Initialize Project Environment
Import libraries for correlation computation and adjacency matrix construction.

In [1]:
from __future__ import annotations

import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

print("pandas", pd.__version__)
print("numpy", np.__version__)

pandas 2.2.3
numpy 2.1.3


## 2. Define Configuration Parameters
Centralize correlation method, thresholding strategy, and soft-thresholding power for WGCNA-style networks.

In [2]:
@dataclass
class NetworkConfig:
    input_file: Path = Path("artifacts/task1_expression_preprocessed.csv")
    export_dir: Path = Path("artifacts")
    corr_method: str = "pearson"
    use_abs_correlation: bool = True
    hard_threshold: Optional[float] = None
    soft_power: int = 6
    use_soft_threshold: bool = True

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["input_file"] = str(info["input_file"])
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = NetworkConfig()
CONFIG.describe()

{'input_file': 'artifacts/task1_expression_preprocessed.csv',
 'export_dir': 'artifacts',
 'corr_method': 'pearson',
 'use_abs_correlation': True,
 'hard_threshold': None,
 'soft_power': 6,
 'use_soft_threshold': True}

## 3. Load Preprocessed Expression Data

In [3]:
def load_expression(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, index_col=0)
    logging.info(
        f"Loaded expression matrix: {df.shape[0]} genes x {df.shape[1]} samples"
    )
    return df


expr_data = load_expression(CONFIG.input_file)
expr_data.head()

[INFO] Loaded expression matrix: 2001 genes x 181 samples


,GSM1213669,GSM1213670,GSM1213671,GSM1213672,GSM1213673,GSM1213674,GSM1213675,GSM1213676,GSM1213677,GSM1213678,...,GSM1213840,GSM1213841,GSM1213842,GSM1213843,GSM1213844,GSM1213845,GSM1213846,GSM1213847,GSM1213848,GSM1213849
Gene,,,,,,,,,,,,,,,,,,,,,
KRT6A,12.748584,7.600682,2.522234,3.447513,5.289046,12.484240,12.487501,3.166682,12.310659,7.223532,...,2.805136,13.205127,10.422477,12.365052,7.767332,5.447799,9.245193,2.882689,12.446786,10.500599
SPRR1B,9.031516,9.583801,3.511168,8.656000,3.540976,11.032316,11.894948,3.646791,10.078840,3.445245,...,3.202791,13.705600,8.037607,9.329184,10.089060,9.008952,10.952269,4.466389,9.148314,8.220998
SCGB1A1,11.270704,6.439259,11.908472,3.578830,3.733639,6.615157,6.551806,4.529208,3.324227,9.495581,...,5.133556,4.100377,9.786214,4.064680,4.383852,5.836540,11.427470,8.748188,8.060045,9.608806
RPS4Y1,11.414617,4.690488,11.788965,4.540654,10.699047,11.721166,5.859753,8.783656,4.480617,4.657880,...,11.376956,9.229483,4.401298,12.263688,10.490783,6.078063,4.740944,4.497662,4.368025,4.180693
SPINK1,5.698196,11.904706,9.032240,7.741032,7.089012,4.170886,3.323415,3.509905,3.562707,12.295183,...,4.680854,7.370971,11.072242,5.451755,9.895093,7.162684,7.385929,6.839303,6.015390,11.157566


## 4. Compute Correlation Matrix

In [4]:
def compute_correlation_matrix(
    df: pd.DataFrame, method: str = "pearson", use_abs: bool = True
) -> pd.DataFrame:
    corr = df.T.corr(method=method)
    if use_abs:
        corr = corr.abs()
        logging.info(f"Using absolute {method} correlation (unsigned network)")
    np.fill_diagonal(corr.values, 1.0)
    logging.info(f"Correlation matrix shape: {corr.shape}")
    return corr


corr_matrix = compute_correlation_matrix(
    expr_data, method=CONFIG.corr_method, use_abs=CONFIG.use_abs_correlation
)
corr_matrix.iloc[:6, :6]

[INFO] Using absolute pearson correlation (unsigned network)
[INFO] Correlation matrix shape: (2001, 2001)


Gene,KRT6A,SPRR1B,SCGB1A1,RPS4Y1,SPINK1,KRT5
Gene,,,,,,
KRT6A,1.000000,0.648675,0.046442,0.080130,0.401516,0.822092
SPRR1B,0.648675,1.000000,0.022699,0.101733,0.199386,0.582058
SCGB1A1,0.046442,0.022699,1.000000,0.021277,0.245187,0.035083
RPS4Y1,0.080130,0.101733,0.021277,1.000000,0.082760,0.133628
SPINK1,0.401516,0.199386,0.245187,0.082760,1.000000,0.477796
KRT5,0.822092,0.582058,0.035083,0.133628,0.477796,1.000000


In [5]:
corr_values = corr_matrix.values[np.triu_indices(len(corr_matrix), k=1)]
corr_stats = {
    "n_pairs": len(corr_values),
    "mean": corr_values.mean(),
    "std": corr_values.std(),
}
pd.DataFrame([corr_stats])

,n_pairs,mean,std
0,2001000,0.17262,0.139394


## 5. Construct Adjacency Matrix

In [6]:
def soft_threshold_adjacency(corr: pd.DataFrame, power: int = 6) -> pd.DataFrame:
    adj = corr**power
    np.fill_diagonal(adj.values, 0)
    adj_vals = adj.values[np.triu_indices(len(adj), k=1)]
    logging.info(
        f"Soft-threshold adjacency (power={power}): mean={adj_vals.mean():.6f}"
    )
    return adj


def hard_threshold_adjacency(
    corr: pd.DataFrame, threshold: float = 0.7
) -> pd.DataFrame:
    adj = (corr >= threshold).astype(int)
    np.fill_diagonal(adj.values, 0)
    n_edges = (adj.values > 0).sum() // 2
    logging.info(f"Hard-threshold adjacency (threshold={threshold}): {n_edges} edges")
    return adj


if CONFIG.use_soft_threshold:
    adjacency = soft_threshold_adjacency(corr_matrix, CONFIG.soft_power)
    threshold_method = f"soft (power={CONFIG.soft_power})"
else:
    adjacency = hard_threshold_adjacency(corr_matrix, CONFIG.hard_threshold or 0.7)
    threshold_method = f"hard (threshold={CONFIG.hard_threshold or 0.7})"

adjacency.iloc[:6, :6]

[INFO] Soft-threshold adjacency (power=6): mean=0.002714


Gene,KRT6A,SPRR1B,SCGB1A1,RPS4Y1,SPINK1,KRT5
Gene,,,,,,
KRT6A,0.000000e+00,7.450121e-02,1.003393e-08,2.647113e-07,4.190016e-03,3.086896e-01
SPRR1B,7.450121e-02,0.000000e+00,1.367995e-10,1.108621e-06,6.282983e-05,3.888640e-02
SCGB1A1,1.003393e-08,1.367995e-10,0.000000e+00,9.278353e-11,2.172599e-04,1.864660e-09
RPS4Y1,2.647113e-07,1.108621e-06,9.278353e-11,0.000000e+00,3.213041e-07,5.693692e-06
SPINK1,4.190016e-03,6.282983e-05,2.172599e-04,3.213041e-07,0.000000e+00,1.189743e-02
KRT5,3.086896e-01,3.888640e-02,1.864660e-09,5.693692e-06,1.189743e-02,0.000000e+00


In [7]:
def pick_soft_threshold(corr: pd.DataFrame, powers: list = None) -> pd.DataFrame:
    if powers is None:
        powers = [2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16]
    results = []
    for power in powers:
        adj = corr**power
        np.fill_diagonal(adj.values, 0)
        connectivity = adj.sum(axis=1)
        mean_conn = connectivity.mean()
        results.append({"power": power, "mean_connectivity": mean_conn})
    return pd.DataFrame(results)


power_analysis = pick_soft_threshold(corr_matrix)
power_analysis

,power,mean_connectivity
0,2,98.456646
1,3,37.264990
2,4,17.151327
3,5,9.127541
4,6,5.427138
5,7,3.515744
6,8,2.434650
7,9,1.776395
8,10,1.350608
9,12,0.855871


## 6. Validate with Unit Tests

In [8]:
def test_correlation_symmetry():
    assert np.allclose(corr_matrix.values, corr_matrix.values.T), (
        "Correlation matrix not symmetric"
    )


def test_correlation_range():
    assert corr_matrix.values.min() >= 0, "Negative correlation in absolute matrix"
    assert corr_matrix.values.max() <= 1, "Correlation > 1"


def test_adjacency_no_self_loops():
    assert np.allclose(np.diag(adjacency.values), 0), "Self-loops present"


def test_soft_threshold():
    test_corr = pd.DataFrame(
        [[1.0, 0.8], [0.8, 1.0]], index=["A", "B"], columns=["A", "B"]
    )
    test_adj = soft_threshold_adjacency(test_corr, power=2)
    assert np.isclose(test_adj.loc["A", "B"], 0.64), (
        f"Expected ~0.64, got {test_adj.loc['A', 'B']}"
    )


test_correlation_symmetry()
test_correlation_range()
test_adjacency_no_self_loops()
test_soft_threshold()
print("All network construction tests passed.")

[INFO] Soft-threshold adjacency (power=2): mean=0.640000


All network construction tests passed.


## 7. Export Results

In [9]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

corr_matrix.to_csv(EXPORT_DIR / "task2_correlation_matrix.csv")
print(
    f"[OK] Correlation matrix saved to: {EXPORT_DIR / 'task2_correlation_matrix.csv'}"
)

adjacency.to_csv(EXPORT_DIR / "task2_adjacency_matrix.csv")
print(f"[OK] Adjacency matrix saved to: {EXPORT_DIR / 'task2_adjacency_matrix.csv'}")

power_analysis.to_csv(EXPORT_DIR / "task2_power_analysis.csv", index=False)
print(f"[OK] Power analysis saved to: {EXPORT_DIR / 'task2_power_analysis.csv'}")

params = {
    "correlation_method": CONFIG.corr_method,
    "threshold_method": threshold_method,
    "n_genes": len(corr_matrix),
}
pd.DataFrame([params]).to_csv(EXPORT_DIR / "task2_network_params.csv", index=False)
print(f"[OK] Network parameters saved to: {EXPORT_DIR / 'task2_network_params.csv'}")

[OK] Correlation matrix saved to: artifacts/task2_correlation_matrix.csv
[OK] Adjacency matrix saved to: artifacts/task2_adjacency_matrix.csv
[OK] Power analysis saved to: artifacts/task2_power_analysis.csv
[OK] Network parameters saved to: artifacts/task2_network_params.csv
